# KrillionSolver Llama 3.1 8B training
In Colab, choose **Runtime > Change runtime type > T4 GPU**. Run each cell in order. The free tier can disconnect or limit GPU availability, so keep the downloaded export when training finishes.

In [ ]:
!pip install -q unsloth datasets trl

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile

uploaded = files.upload()
bundle_name = next((name for name in uploaded if name.endswith('.zip')), None)
if not bundle_name:
    raise ValueError('Upload KrillionSolver-Colab-Training.zip')
work = Path('/content/krillion')
if work.exists():
    shutil.rmtree(work)
work.mkdir()
with zipfile.ZipFile(bundle_name) as bundle:
    bundle.extractall(work)
print('Loaded training bundle into', work)

In [ ]:
import os, runpy, torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is active. Select Runtime > Change runtime type > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
os.environ['LOCALAPPDATA'] = '/content/krillion/user_data'
os.environ['KRILLION_OUTPUT_DIR'] = '/content/krillion/training/outputs'
os.environ['KRILLION_EXPORT_DIR'] = '/content/krillion/training/export'
runpy.run_path('/content/krillion/training/finetune_krillion.py', run_name='__main__')

In [ ]:
from google.colab import files
import shutil
from pathlib import Path
export_dir = Path('/content/krillion/training/export_gguf')
if not export_dir.is_dir():
    export_dir = Path('/content/krillion/training/export')
if not list(export_dir.glob('*.gguf')):
    raise FileNotFoundError(f'No GGUF model found in {export_dir}')
archive = shutil.make_archive('/content/KrillionSolver-trained-model', 'zip', export_dir)
print('Downloading', archive)
files.download(archive)

After downloading, extract the ZIP on your PC. From the extracted folder, run `ollama create krillionsolver:8b -f Modelfile`. Then restart KrillionSolver; its launcher will select that tag automatically.